# Tachyons and the antitelephone

*cosmology · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/tachyons)

What if a particle had an imaginary mass, $m^2 < 0$? With complex tensors, $E^2 = p^2 + m^2$ still gives real energies, but the particle is always faster than light and speeds up as it loses energy. Batched Lorentz boosts of superluminal worldlines find frames where the signal arrives before it is sent, and close Tolman's antitelephone loop. In field theory $m^2 < 0$ instead means an unstable vacuum that condenses, as the Higgs field does. Speculative: no tachyon has ever been seen, and the 2011 OPERA "faster-than-light neutrinos" were a loose cable.

**Vector form**

$$E^2 = p^2c^2 + m^2c^4,\quad m^2 = -\mu^2 < 0\;\Rightarrow\; v = c\sqrt{1 + \frac{\mu^2c^4}{E^2}} > c,\qquad \Delta t' = \gamma\,\Delta t\Big(1 - \frac{\beta\,u}{c}\Big) < 0 \;\text{ if }\; \beta c > \frac{c^2}{u}$$

**Index (tensor) form**

$$x'^a_{Bn} = \Lambda^a{}_b(\beta_B)\,x^b_n,\qquad p^a\eta_{ab}p^b = -\mu^2,\qquad t_{\mathrm{back}} = t_{\mathrm{send}}\,\frac{u^2(1-\beta^2)}{(u-\beta)^2},\qquad \ddot\phi_i + \gamma\dot\phi_i = \frac{\phi_{i+1} - 2\phi_i + \phi_{i-1}}{\Delta x^2} - m^2\phi_i - \lambda\phi_i^3$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $E = \sqrt{p^2 + m^2}$ | `energy(p, 1j * mu)` | (U,) complex | complex square root: real when $p > \mu$, even though $m$ and $\gamma$ are both imaginary |
| $\Lambda^a{}_b(\beta_B)$ | `boost_matrix(betas)` | (F, 2, 2) | one Lorentz boost per observer frame ($c = 1$) |
| $x'^a_{Bn}$ | `einsum('Bab,...b->B...a', lam, four)` | (F, U, 2) | repeated $b$ ⇒ summed: every signal seen from every frame at once |
| $p^a\eta_{ab}p^b$ | `einsum('...a,a,...a->...', p, eta, p)` | (F, U) | the invariant $m^2 = -\mu^2$, the same in every frame |
| $u' = (u-\beta)/(1-u\beta)$ | `velocity_add(u, beta)` | broadcast | rapidities subtract: $\tanh$ below $c$, $\coth$ above |
| $t_{\mathrm{back}}$ | `antitelephone(u, betas)` | (F,) | when the reply reaches Alice; $< t_{\mathrm{send}}$ closes a causal loop |
| $\phi_i$ | `condense(phi0, m2, lam, ...)` | (S, N) | lattice field rolling off the false vacuum $\phi = 0$ (torch.roll Laplacian, velocity Verlet) |

Speculative physics: tachyons are hypothetical. Nothing in the Standard Model is a tachyonic particle and no experiment has ever detected one. Special relativity forbids accelerating a massive body through $c$, but it does not by itself forbid particles that are always faster than light (Bilaniuk, Deshpande and Sudarshan 1962; Feinberg 1967, who coined the name). The price is causality. Two events joined by a signal of speed $u > c$ are spacelike-separated, so their time order depends on the observer: in any frame moving faster than $c^2/u$ the signal arrives before it is sent and carries negative energy. Feinberg's reinterpretation reads that as a positive-energy antitachyon travelling the other way, which saves energy positivity but leaves "who sent it" frame-dependent. Using two frames gives Tolman's antitelephone (1917): Alice signals Bob, who is receding at $\beta$, and Bob replies at the same speed in his own frame. The reply reaches Alice before she sent the message once $\beta > 2u/(1+u^2)$, which closes a causal loop. In quantum field theory a negative $m^2$ is not a fast particle but an instability. The field rolls off the hilltop $\phi = 0$ of $V = \tfrac12 m^2\phi^2 + \tfrac14\lambda\phi^4$ into $\phi = \pm\sqrt{-m^2/\lambda}$ ("tachyon condensation", lowering the energy by $m^4/4\lambda$), where ordinary particles of mass$^2 = 2|m^2|$ live. This is exactly what the Higgs field does, and the panel shows it on a lattice with Hubble-like friction. Real data: in 2011 OPERA reported muon neutrinos arriving $60.7 \pm 6.9 \pm 7.4$ ns early over 730 km, $(v-c)/c = (2.48 \pm 0.28 \pm 0.30)\times10^{-5}$ (arXiv:1109.4897v1). The cause turned out to be a badly connected fibre-optic cable plus a clock drift. The corrected result is $(2.7 \pm 3.1\,^{+3.4}_{-3.3})\times 10^{-6}$ (v4, JHEP 10 (2012) 093), and ICARUS measured $\delta t = 0.3 \pm 4.9 \pm 9.0$ ns (Phys. Lett. B 713 (2012) 17): both consistent with $c$. Read as a tachyon, the 2011 value would need $|m| \approx 120$ MeV, and then the 10 MeV neutrinos of supernova SN1987A would have outrun light by a factor of about 12, instead of arriving within hours of it after 168 000 years ($|v-c|/c \lesssim 2\times10^{-9}$, Longo 1987). Units: $c = 1$, the tachyon mass $\mu = 1$ for the signal fan.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/tachyons/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    u0, beta_max = params["u"], params["beta_max"]
    speeds = tensors.signal_speeds(u0, dev)  # (U,)
    betas = tensors.frames(beta_max, 90, dev)  # (F,)

    # 1. imaginary mass: E² = p² + m² with m = iμ (μ = 1 here)
    m = torch.tensor(1j, dtype=torch.complex128, device=dev)
    p = speeds / torch.sqrt(speeds**2 - 1)  # momentum that gives each speed when μ = 1
    e = equation.energy(p, m)
    four = torch.stack([e.real, p], dim=-1)  # (U, 2)
    m2 = equation.minkowski_square(equation.boost(four, betas))  # (F, U)
    print(f"signal speeds u = {speeds.tolist()} c,  energies E = {[round(x, 3) for x in e.real.tolist()]} μ")
    print(f"invariant m² over all {len(betas)} frames: {m2.min().item():.6f} … {m2.max().item():.6f} (= −μ²)")

    # 2. the same signals seen from every frame: arrival before emission when β > 1/u
    events = tensors.emission_and_reception(speeds)  # (U, 2)
    seen = equation.boost(events, betas)  # (F, U, 2)
    energy_seen = equation.boost(four, betas)[..., 0]  # (F, U)
    feinberg = solve.reinterpret(seen[..., 0], energy_seen)
    for u in speeds.tolist():
        print(f"u = {u:5.2f} c: time order flips for β > 1/u = {1 / u:.3f}")
    assert torch.equal(feinberg["emitter_is_receiver"], seen[..., 0] < 0)

    # 3. Tolman's antitelephone with Bob receding at β
    loop = equation.antitelephone(torch.tensor(u0, dtype=torch.float64, device=dev), betas, t_send=1.0)
    threshold = 2 * u0 / (1 + u0**2)  # β above which u > (1 + √(1 − β²))/β
    print(f"antitelephone with u = {u0:g} c: the reply beats the message for β > {threshold:.3f}")
    print(f"at β = {beta_max:g}: reply returns at t = {loop['t_back'][-1].item():.3f} (message sent at t = 1)")

    # 4. the OPERA anomaly read as a tachyon, against SN1987A
    data = tensors.load_measurements()
    opera = data["OPERA (first result)"]
    mu = equation.mass_from_speed(opera["energy_gev"], opera["dv_over_c"])
    sn = data["SN1987A"]
    predicted = equation.speed_excess(sn["energy_gev"], mu)
    print(f"OPERA 2011 as a tachyon: |m| = {1000 * mu:.0f} MeV; it would make 10 MeV SN1987A neutrinos")
    print(f"  (v − c)/c = {predicted:.1f} instead of < {sn['err_plus']:.0e}: ruled out (and the anomaly was an error)")

    # 5. tachyon condensation: a field with m² < 0 rolls off φ = 0
    phi0 = tensors.noise_field(256, 1e-3, seed=0, device=dev)
    field = solve.condense(phi0, params["m2"], params["lam"], 60.0, 40.0, 0.02, params["friction"])
    v = equation.vacuum(params["m2"], params["lam"])
    print(
        f"condensation: true vacuum |φ| = {v:.3f}, V_min = {-(params['m2'] ** 2) / (4 * params['lam']):.3f} < V(0) = 0;"
    )
    print(f"  final ⟨|φ|⟩ = {field['phi'][-1].abs().mean().item():.3f}, ⟨V⟩ = {field['potential'][-1].item():.3f}")
    return plot.animate(speeds, betas, seen, u0, loop, data, mu, field, v, params, out_dir / "tachyons.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/tachyons/equation.py
r"""Physics → tensor translation (units with c = 1).

A tachyon is a hypothetical particle with imaginary mass m = iμ, so m² = −μ² < 0. The mass shell keeps its form

    E² = p² + m² = p² − μ²,      v = p/E = \sqrt{1 + μ²/E²} > 1,      E = mγ with γ = 1/\sqrt{1 − v²} imaginary,

so a tachyon speeds up as it loses energy (v → ∞ as E → 0). Lorentz boosts act on event and momentum
two-vectors alike,

    x'^a = Λ^a{}_b(β)\, x^b,      Λ(β) = γ_β \begin{pmatrix}1 & −β\\ −β & 1\end{pmatrix},

and keep the invariant x^a η_{ab} x^b. A signal of speed u > 1 sent from the origin arrives at t' = γ_β(1 − βu)/u:
earlier than it was sent whenever β > 1/u (= c²/u). In field theory a tachyonic mass m² < 0 is not a fast particle
but an unstable vacuum: φ = 0 is a hill of V(φ) = ½m²φ² + ¼λφ⁴ and the field rolls to φ = ±\sqrt{−m²/λ}.
"""

import torch


def energy(p: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    r"""E = \sqrt{p² + m²} with a complex mass (m = iμ for a tachyon).   → complex, same shape as p"""
    return torch.sqrt(p.to(torch.complex128) ** 2 + m**2)


def velocity(p: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""v = p/E (group velocity dE/dp, the same for any mass)."""
    return p / e


def lorentz_factor(v: torch.Tensor) -> torch.Tensor:
    r"""γ = 1/\sqrt{1 − v²}: real below c, imaginary above (complex input)."""
    return 1 / torch.sqrt(1 - v**2)


def boost_matrix(beta: torch.Tensor) -> torch.Tensor:
    r"""Λ^a{}_b(β) for a batch of frame velocities.   beta: (B,) → (B, 2, 2)"""
    g = 1 / torch.sqrt(1 - beta**2)
    row0 = torch.stack([g, -g * beta], dim=-1)
    row1 = torch.stack([-g * beta, g], dim=-1)
    return torch.stack([row0, row1], dim=-2)


def boost(four: torch.Tensor, beta: torch.Tensor) -> torch.Tensor:
    r"""x'^a = Λ^a{}_b x^b  (b summed) for every frame.

    four: (..., 2) as (t, x) or (E, p), beta: (B,) → (B, ..., 2)
    """
    return torch.einsum("Bab,...b->B...a", boost_matrix(beta), four)


def minkowski_square(four: torch.Tensor) -> torch.Tensor:
    r"""x^a η_{ab} x^b = t² − x² (or E² − p² = m²), η = diag(1, −1)."""
    eta = torch.tensor([1.0, -1.0], dtype=four.dtype, device=four.device)
    return torch.einsum("...a,a,...a->...", four, eta, four)


def velocity_add(u: torch.Tensor, beta: torch.Tensor) -> torch.Tensor:
    r"""Speed u seen from a frame moving at β:  u' = (u − β)/(1 − uβ)  (rapidities subtract)."""
    return (u - beta) / (1 - u * beta)


def antitelephone(u: torch.Tensor, beta: torch.Tensor, t_send: float = 1.0) -> dict:
    r"""Tolman's antitelephone. Bob moves away at β (worldline x = βt). Alice sends a tachyon of speed u at t_send;
    Bob answers at once with a tachyon of speed u *in his own frame*. Returns the reception event and the time the
    answer gets back to Alice; t_back < t_send closes a causal loop.   u, beta broadcast together.
    """
    t_rec = u * t_send / (u - beta)  # u (t − t_send) = β t
    x_rec = beta * t_rec
    w = velocity_add(-u, -beta)  # Bob's −u in Alice's frame: (β − u)/(1 − βu)
    t_back = t_rec - x_rec / w  # where the reply worldline meets x = 0
    return {"t_receive": t_rec, "x_receive": x_rec, "w": w, "t_back": t_back}


def potential(phi: torch.Tensor, m2: float, lam: float) -> torch.Tensor:
    r"""V(φ) = ½ m² φ² + ¼ λ φ⁴; with m² < 0 the top φ = 0 is a false vacuum."""
    return 0.5 * m2 * phi**2 + 0.25 * lam * phi**4


def vacuum(m2: float, lam: float) -> float:
    r"""True vacuum |φ| = \sqrt{−m²/λ} after tachyon condensation."""
    return (-m2 / lam) ** 0.5


def force(phi: torch.Tensor, m2: float, lam: float, dx: float) -> torch.Tensor:
    r"""−δH/δφ = ∂²φ/∂x² − m²φ − λφ³ on a periodic lattice (torch.roll for the neighbours).   (..., N)"""
    lap = (torch.roll(phi, 1, -1) - 2 * phi + torch.roll(phi, -1, -1)) / dx**2
    return lap - m2 * phi - lam * phi**3


def mass_from_speed(energy_gev: float, dv_over_c: float) -> float:
    r"""|m| = E\sqrt{v² − 1} of the tachyon that would have speed v = 1 + δv at energy E."""
    v = 1 + dv_over_c
    return energy_gev * (v**2 - 1) ** 0.5


def speed_excess(energy_gev, mu: float):
    r"""(v − c)/c = \sqrt{1 + μ²/E²} − 1 for a tachyon of mass |m| = μ at energy E (float or tensor)."""
    return (1 + (mu / energy_gev) ** 2) ** 0.5 - 1


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/tachyons/tensors.py
"""Build the concrete tensors: tachyon signals, observer frames, the neutrino-speed table and a noisy field.

data/neutrino_speed.csv holds published values of (v − c)/c: the 2011 OPERA anomaly (arXiv:1109.4897v1), the
corrected OPERA result (v4, JHEP 10 (2012) 093), ICARUS (Phys. Lett. B 713 (2012) 17) and the SN1987A bound.
"""

from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "neutrino_speed.csv"
DTYPE = torch.float64


def signal_speeds(u: float, device=None) -> torch.Tensor:
    """A fan of tachyon speeds around the chosen one (units of c): (U,)."""
    return torch.tensor([1.25, 2.0, u, 6.0, 25.0], dtype=DTYPE, device=device).unique()


def frames(beta_max: float, n: int, device=None) -> torch.Tensor:
    """Observer velocities 0 … β_max swept by the animation: (F,)."""
    return torch.linspace(0.0, beta_max, n, dtype=DTYPE, device=device)


def emission_and_reception(u: torch.Tensor, distance: float = 1.0) -> torch.Tensor:
    """Reception events (t = d/u, x = d) of signals sent from the origin: (U, 2) as (t, x)."""
    return torch.stack([distance / u, torch.full_like(u, distance)], dim=-1)


def noise_field(n: int, amplitude: float, seed: int = 0, device=None) -> torch.Tensor:
    """Tiny Gaussian fluctuations around the false vacuum φ = 0 on n lattice sites: (N,)."""
    gen = torch.Generator().manual_seed(seed)
    return (amplitude * torch.randn(n, generator=gen, dtype=DTYPE)).to(device)


def load_measurements(path: Path = DATA) -> dict:
    """{experiment: {year, energy_gev, dv_over_c, err_minus, err_plus, kind}} from the bundled CSV."""
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    header = lines[0].split(",")
    out = {}
    for ln in lines[1:]:
        row = dict(zip(header, ln.split(",")))
        name = row.pop("experiment")
        out[name] = {k: (v if k == "kind" else float(v)) for k, v in row.items()}
    return out


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/tachyons/solve.py
"""Feinberg's reinterpretation of backward-in-time signals, and tachyon condensation on a lattice."""

import torch

from problems._common.progress import steps as progress_steps

from . import equation


def reinterpret(dt: torch.Tensor, energy: torch.Tensor) -> dict:
    """Feinberg (1967): a tachyon seen arriving before it left carries negative energy; read it instead as a
    positive-energy antitachyon sent by the receiver to the emitter, forward in time. No observer then sees negative
    energy, but who 'sent' the signal depends on the frame, which is why the antitelephone is still a paradox.
    """
    backward = dt < 0
    return {"emitter_is_receiver": backward & (energy < 0), "energy": energy.abs(), "duration": dt.abs()}


def condense(phi0, m2: float, lam: float, length: float, t_max: float, dt: float, friction: float = 0.0) -> dict:
    """φ_tt + γ φ_t = φ_xx − m²φ − λφ³ from tiny noise, velocity Verlet with the friction as an exact damping factor.

    phi0: (N,) → dict with phi (S+1, N), energy, kinetic and mean potential (S+1,).
    The friction γ plays the role of Hubble friction 3H in an expanding universe.
    """
    n = phi0.shape[-1]
    dx = length / n
    steps = int(round(t_max / dt))
    damp = torch.exp(torch.tensor(-friction * dt / 2, dtype=phi0.dtype))
    phi, pi = phi0.clone(), torch.zeros_like(phi0)
    traj, energy, kinetic, pot = [], [], [], []

    def record():
        grad = (torch.roll(phi, -1, -1) - phi) / dx
        kin = 0.5 * (pi**2).sum() * dx
        v = equation.potential(phi, m2, lam)
        traj.append(phi.clone())
        kinetic.append(kin)
        energy.append(kin + (0.5 * grad**2 + v).sum() * dx)
        pot.append(v.mean())

    record()
    for _ in progress_steps(steps, "lattice"):
        pi = damp * pi + 0.5 * dt * equation.force(phi, m2, lam, dx)
        phi = phi + dt * pi
        pi = damp * (pi + 0.5 * dt * equation.force(phi, m2, lam, dx))
        record()
    return {
        "phi": torch.stack(traj),
        "energy": torch.stack(energy),
        "kinetic": torch.stack(kinetic),
        "potential": torch.stack(pot),
        "x": torch.arange(n, dtype=phi0.dtype) * dx,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/tachyons/plot.py
"""Four panels: tachyon signals in a moving frame, the antitelephone loop, neutrino-speed data and condensation."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, GREY, AMBER, GREEN = "#2563eb", "#dc2626", "#9ca3af", "#f59e0b", "#16a34a"


def animate(speeds, betas, seen, u0, loop, data, mu, field, v, params, path: Path) -> Path:
    speeds, betas, seen = speeds.cpu().numpy(), betas.cpu().numpy(), seen.cpu().numpy()  # (U,), (F,), (F, U, 2)
    t_rec, x_rec = loop["t_receive"].cpu().numpy(), loop["x_receive"].cpu().numpy()
    t_back = loop["t_back"].cpu().numpy()
    phi, x = field["phi"].cpu().numpy(), field["x"].cpu().numpy()
    pot = field["potential"].cpu().numpy()

    fig, axes = plt.subplots(2, 2, figsize=(9, 6.8))
    ax_m, ax_a, ax_d, ax_c = axes.ravel()

    # A. Minkowski diagram of the signal fan in the moving frame
    lim = 1.6
    ax_m.plot([-lim, lim], [-lim, lim], color=GREY, ls="--", lw=1)
    ax_m.plot([-lim, lim], [lim, -lim], color=GREY, ls="--", lw=1)
    ax_m.set_xlim(-lim, lim)
    ax_m.set_ylim(-lim, lim)
    ax_m.set_aspect("equal")
    ax_m.set_xlabel("x′")
    ax_m.set_ylabel("t′")
    lines = [ax_m.plot([], [], lw=2)[0] for _ in speeds]
    dots = ax_m.scatter([], [], s=18, zorder=3)
    labels = [ax_m.text(0, 0, f"{u:g}c", fontsize=7) for u in speeds]
    title_m = ax_m.set_title("signals seen at β\nred: received before sent", fontsize=9)

    # B. antitelephone in Alice's frame
    t_top = min(max(1.6, float(np.max(t_rec)) * 1.1), 8.0)
    t_bottom = min(-0.3, float(np.min(t_back)) - 0.2)
    ax_a.set_xlim(-0.2, max(1.0, float(np.max(x_rec[t_rec < t_top])) * 1.15))
    ax_a.set_ylim(max(t_bottom, -3.0), t_top)
    ax_a.axvline(0, color="#374151", lw=2)
    ax_a.axhline(1.0, color=AMBER, lw=1, ls=":")
    ax_a.text(0.02, 1.02, "message sent", color="#b45309", fontsize=7, transform=ax_a.get_yaxis_transform())
    (bob,) = ax_a.plot([], [], color="#374151", lw=1.5, ls="--")
    (msg,) = ax_a.plot([], [], color=BLUE, lw=2)
    (reply,) = ax_a.plot([], [], color=RED, lw=2)
    back_dot = ax_a.scatter([0], [1], s=30, color=RED, zorder=4)
    status = ax_a.text(0.97, 0.04, "", transform=ax_a.transAxes, ha="right", fontsize=8)
    title_a = ax_a.set_title("antitelephone\n", fontsize=9)
    ax_a.set_xlabel("x (Alice's frame)")
    ax_a.set_ylabel("t")
    loop_fill = []

    # C. neutrino speed data (static)
    names = ["OPERA (first result)", "OPERA (corrected)", "ICARUS", "SN1987A"]
    ys = np.arange(len(names))[::-1]
    for y, name in zip(ys, names):
        row = data[name]
        colour = RED if name == "OPERA (first result)" else (GREEN if row["kind"] == "bound" else BLUE)
        ax_d.errorbar(
            1e6 * row["dv_over_c"],
            y,
            xerr=[[1e6 * row["err_minus"]], [1e6 * row["err_plus"]]],
            fmt="o",
            color=colour,
            capsize=3,
        )
        ax_d.text(-14, y + 0.18, f"{name} {int(row['year'])}", fontsize=7)
    ax_d.axvline(0, color="#374151", lw=1)
    ax_d.set_xlim(-15, 32)
    ax_d.set_ylim(-0.6, len(names) - 0.3)
    ax_d.set_yticks([])
    ax_d.set_xlabel(r"$(v-c)/c$  ($10^{-6}$)")
    ax_d.set_title("measured neutrino speeds: all consistent with c", fontsize=9)
    ax_d.text(
        0.98,
        0.04,
        f"OPERA 2011 as a tachyon: |m| ≈ {1000 * mu:.0f} MeV\n"
        f"→ 10 MeV SN1987A ν at ≈ {(1 + (mu / 0.01) ** 2) ** 0.5:.0f}c: ruled out",
        transform=ax_d.transAxes,
        ha="right",
        fontsize=7,
        color=RED,
    )

    # D. tachyon condensation
    ax_c.axhline(v, color=GREEN, ls="--", lw=1)
    ax_c.axhline(-v, color=GREEN, ls="--", lw=1)
    ax_c.axhline(0, color=GREY, lw=0.8)
    (field_line,) = ax_c.plot([], [], color=BLUE, lw=1.5)
    ax_c.set_xlim(x[0], x[-1])
    ax_c.set_ylim(-1.8 * v, 1.8 * v)
    ax_c.set_xlabel("x")
    ax_c.set_ylabel("φ")
    title_c = ax_c.set_title("tachyon condensation", fontsize=9)

    n_steps = phi.shape[0]

    def update(f: int):
        beta = betas[f]
        # A
        flipped = seen[f, :, 0] < 0
        scale = lim * 0.9 / np.maximum(np.abs(seen[f]).max(-1), 1e-9)  # draw each signal to the same length
        ends = seen[f] * np.minimum(scale, 1.0)[:, None]
        for k, line in enumerate(lines):
            line.set_data([0, ends[k, 1]], [0, ends[k, 0]])
            line.set_color(RED if flipped[k] else BLUE)
            line.set_linestyle("--" if flipped[k] else "-")
            labels[k].set_position((ends[k, 1] + 0.03, ends[k, 0]))
        dots.set_offsets(np.column_stack([ends[:, 1], ends[:, 0]]))
        dots.set_color([RED if s else BLUE for s in flipped])
        title_m.set_text(f"signals from the origin seen at β = {beta:.2f}\nred: received before sent (β > 1/u)")
        # B
        for item in loop_fill:
            item.remove()
        loop_fill.clear()
        bob.set_data([0, beta * t_top], [0, t_top])
        msg.set_data([0, x_rec[f]], [1.0, t_rec[f]])
        reply.set_data([x_rec[f], 0], [t_rec[f], t_back[f]])
        back_dot.set_offsets([[0, t_back[f]]])
        closed = t_back[f] < 1.0
        loop_fill.append(
            ax_a.fill([0, x_rec[f], 0], [1.0, t_rec[f], t_back[f]], color=RED if closed else BLUE, alpha=0.12)[0]
        )
        status.set_text(
            f"reply {1 - t_back[f]:.2f} before the message!" if closed else f"reply arrives at t = {t_back[f]:.2f}"
        )
        status.set_color(RED if closed else "#111827")
        title_a.set_text(f"antitelephone: Bob recedes at β = {beta:.2f}\nAlice and Bob signal at u = {u0:g}c")
        # D
        step = int(round(f / max(len(betas) - 1, 1) * (n_steps - 1)))
        field_line.set_data(x, phi[step])
        title_c.set_text(f"tachyon condensation (m² = {params['m2']:g}): ⟨V⟩ = {pot[step]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(betas), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'u': 3.0,
    'beta_max': 0.9,
    'm2': -1.0,
    'lam': 1.0,
    'friction': 0.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.tachyons")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.tachyons.main")
problem_dir = Path("problems/cosmology/tachyons")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
